# ML Laboratuvarı · Modül 3, Ünite 9: Enflasyonu öngörmek

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 3

Veri: 13 aylık ABD serisi (FRED). Kurulum hücresi `G` adlı aylık bir tablo hazırlar (indeks: tarih). Sütunlar: `enf` (yıllık enflasyon, %), `aylik` (aylık enflasyon, %), 7 serinin yıllık değişimi (`CPILFESL`, `PPIACO`, `M2SL`, `WTISPLC`, `AHETPI`, `PAYEMS`, `INDPRO`), 4 serinin düzeyi (`UNRATE`, `FEDFUNDS`, `T10Y3MM`, `UMCSENT`) ve `HOUST` (log). Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Başlamadan:** GitHub'dan açılan bu notebook'u düzenleyebilmek için **Dosya → Drive'a kopya kaydet**'i seç ve açılan kopyada çalış.

**Kurallar ve sıra**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku. Gemini kodu yorum satırının altına yeni bir hücre olarak ekleyebilir; sırayla çalıştır.
3. Görev kodu hatasız çalışınca ilgili **Kontrol** hücresini çalıştır. Hata gösterirse düzelt, görev hücresini ve kontrolü yeniden çalıştır.
4. Kontrol geçerse bir **tamamlama kodu** verir; onu ve görev kodunu ML Laboratuvarı'ndaki kutuya yapıştır.

> **Veri kesiti:** Kurulum hücresi indirme zamanını ve her serinin son ayını yazdırır. Canlı FRED verisiyle sayılar Lab'dakilerden biraz farklı olabilir; sabit bir kesitle çalışmak için Kurulum hücresindeki `SON_AY` ayarını kullan (yalnızca o satırı değiştir).
>
> **Geriye dönük deney:** Ekim 2025'in TÜFE verisi yok (ABD federal hükümetinin kapanması); eksik ayları doldurma, içeren satırları dışarıda bırak. Basitlik için “bu ayın verisi ay sonunda biliniyor” varsayıyoruz; gerçekte TÜFE bir sonraki ayın ortasında yayımlanır, diğer serilerin de yayın gecikmeleri ve revizyonları vardır. Gerçek zamanlı bir öngörü, tahmin günü bilinen veri sürümlerini (ALFRED) kullanmalıdır.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, base64, urllib.request, warnings, datetime
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, KFold
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("always", category=ConvergenceWarning)   # yakınsama uyarısı görünür kalır
SON_AY = None   # Sabit veri kesiti için bir ay yaz (ör. "2025-12-01"): bu aydan sonraki gözlemler atılır. None: en güncel veri.

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

kodlar = ['CPIAUCSL', 'CPILFESL', 'PPIACO', 'M2SL', 'WTISPLC', 'AHETPI', 'PAYEMS', 'INDPRO', 'UNRATE', 'FEDFUNDS', 'T10Y3MM', 'UMCSENT', 'HOUST']
print("İndirme zamanı (UTC):", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"))
ham = pd.concat([fred_indir(k).set_index("tarih") for k in kodlar], axis=1).sort_index().asfreq("MS")
if SON_AY is not None:
    ham = ham.loc[:pd.Timestamp(SON_AY)]
    print("Sabit kesit:", SON_AY, "sonrasındaki gözlemler atıldı.")
G = pd.DataFrame(index=ham.index)
G["enf"] = ham["CPIAUCSL"].pct_change(12, fill_method=None) * 100     # yıllık enflasyon (%)
G["aylik"] = ham["CPIAUCSL"].pct_change(1, fill_method=None) * 100    # aylık enflasyon (%)
for k in ["CPILFESL", "PPIACO", "M2SL", "WTISPLC", "AHETPI", "PAYEMS", "INDPRO"]:
    G[k] = ham[k].pct_change(12, fill_method=None) * 100              # yıllık değişim (%)
for k in ["UNRATE", "FEDFUNDS", "T10Y3MM", "UMCSENT"]:
    G[k] = ham[k]                                                      # düzey
G["HOUST"] = np.log(ham["HOUST"])                                      # konut başlangıcı (log)
_G = G.copy()
BASLA, KESIM = pd.Timestamp("1984-01-01"), pd.Timestamp("2015-01-01")   # KESIM: ilk test HEDEF ayı
_REF = {}

def _k(s):
    return base64.b64decode(s).decode()

def _kh(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))

def _referans():
    if _REF:
        return _REF
    oz = {}
    for k in _G.columns:
        if k != "aylik":
            for g in range(3):
                oz[f"{k}_g{g}"] = _G[k].shift(g)
    for g in range(12):
        oz[f"aylik_g{g}"] = _G["aylik"].shift(g)
    X = pd.DataFrame(oz)
    t = pd.concat([X, _G["enf"].shift(-1).rename("hedef")], axis=1).loc[BASLA:].dropna()
    t["hedef_ayi"] = t.index + pd.DateOffset(months=1)
    e, s = t[t["hedef_ayi"] < KESIM], t[t["hedef_ayi"] >= KESIM]
    c = list(X.columns)
    ekk = make_pipeline(StandardScaler(), LinearRegression()).fit(e[c], e["hedef"])
    las = GridSearchCV(make_pipeline(StandardScaler(), Lasso(max_iter=50000)), {"lasso__alpha": np.logspace(-4, 0.5, 60)},
                       cv=TimeSeriesSplit(5), scoring="neg_mean_squared_error").fit(e[c], e["hedef"])
    _REF.update(n_test=len(s), n_egitim=len(e), saf=_kh(s["enf_g0"], s["hedef"]), ekk=_kh(ekk.predict(s[c]), s["hedef"]),
                lasso=_kh(las.predict(s[c]), s["hedef"]), son_satir=t.index.max(), son_hedef=t["hedef_ayi"].max())
    return _REF

def kontrol_A(tablo, egitim, test, kok_hko_saf, kok_hko_ekk):
    if "hedef" not in tablo.columns:
        print("❌ 'tablo' içinde 'hedef' adlı bir sütun bulunamadı."); return
    h = tablo["hedef"]
    if np.allclose(h.values, _G["enf"].reindex(h.index).values, equal_nan=True):
        print("❌ Hedef kaydırılmamış: 'hedef' bu ayın enflasyonu. Gelecek ayın değeri için shift(-1) gerekir (hedef sızıntısı)."); return
    if not np.allclose(h.values, _G["enf"].shift(-1).reindex(h.index).values, equal_nan=True):
        print("❌ 'hedef' gelecek ayın yıllık enflasyonu (enf.shift(-1)) ile uyuşmuyor. Hedef farklı kurulmuş ya da eksik aylar doldurulmuş olabilir."); return
    if "hedef_ayi" not in tablo.columns:
        print("❌ 'hedef_ayi' sütunu yok. Her satırın hedefinin ait olduğu ayı ekle: tablo['hedef_ayi'] = tablo.index + pd.DateOffset(months=1)"); return
    if not (pd.to_datetime(tablo["hedef_ayi"]).values == (tablo.index + pd.DateOffset(months=1)).values).all():
        print("❌ 'hedef_ayi' özellik ayının (indeks) bir ay sonrası olmalı."); return
    if tablo.drop(columns=["hedef_ayi"]).isna().sum().sum() > 0:
        print("❌ Tabloda eksik değer var; eksik ayları içeren satırlar dışarıda bırakılmalı (dropna)."); return
    if len(tablo.columns) - 2 != 51:
        print(f"❌ 51 özellik bekleniyordu, {len(tablo.columns) - 2} var (hedef ve hedef_ayi hariç). Tarifi yeniden oku: 13 seri × 3 gecikme + aylık enflasyonun 12 gecikmesi."); return
    for ad, d in [("egitim", egitim), ("test", test)]:
        if "hedef_ayi" not in d.columns or not d.index.is_monotonic_increasing:
            print(f"❌ '{ad}' tarih sırasında olmalı ve 'hedef_ayi' sütununu içermeli; karıştırma olmamalı."); return
    if egitim["hedef_ayi"].max() >= KESIM:
        print(f"❌ Eğitimde hedefi {egitim['hedef_ayi'].max():%Y-%m} olan bir satır var. Bölme HEDEF ayına göre yapılmalı: Aralık 2014 satırının hedefi Ocak 2015'tir ve teste aittir."); return
    if test["hedef_ayi"].min() < KESIM:
        print("❌ Testte hedefi 2015 öncesine ait satır var; test, hedef ayı 2015-01 ve sonrası olmalı."); return
    r = _referans()
    if len(test) > r["n_test"]:
        print("❌ Test döneminde beklenenden fazla ay var: eksik aylar doldurulmuş olabilir. Doldurma; dışarıda bırak."); return
    if len(test) < r["n_test"] or len(egitim) < r["n_egitim"]:
        print(f"❌ Beklenenden az satır var (eğitim {len(egitim)}/{r['n_egitim']}, test {len(test)}/{r['n_test']}). Kesim ya da başlangıç tarihi farklı olabilir."); return
    if egitim.index.min() > BASLA + pd.Timedelta(days=40):
        print("❌ Eğitim verisi 1984'ten başlamıyor."); return
    if abs(float(kok_hko_saf) - r["saf"]) > 0.005:
        print("❌ Saf öngörünün kök HKO'su beklenenden farklı. Öngörü 'bu ayın yıllık enflasyonu' (enf_g0) mı, hedefle mi karşılaştırıldı?"); return
    if abs(float(kok_hko_ekk) - r["ekk"]) > 0.02:
        print("❌ EKK'nın test kök HKO'su beklenenden farklı. 51 özelliğin hepsi kullanıldı mı? Model yalnızca eğitimle mi kuruldu?"); return
    print(f"Son eğitim hedefi: {egitim['hedef_ayi'].max():%Y-%m} · ilk test hedefi: {test['hedef_ayi'].min():%Y-%m} · test satırı: {len(test)}")
    print(f"Veri kesiti: son tam özellik satırı {r['son_satir']:%Y-%m}, son hedef ayı {r['son_hedef']:%Y-%m}")
    print(f"✅ Tüm kontroller geçti (saf ≈ {r['saf']:.3f}, EKK ≈ {r['ekk']:.3f}). Tamamlama kodun: " + _k("TUwzLUVORi00S1Fa"))

def kontrol_B(arama, kok_hko_lasso, secilenler, egitim, test):
    if not isinstance(arama, GridSearchCV):
        print("❌ 'arama' bir GridSearchCV nesnesi olmalı. LassoCV'ye önceden ölçeklenmiş veri vermek, her doğrulama katının kendi ortalama ve sapmasını ölçeklemeye taşır; Pipeline(StandardScaler, Lasso) + GridSearchCV kullan."); return
    est = arama.estimator
    if not (isinstance(est, Pipeline) and isinstance(est.steps[0][1], StandardScaler) and isinstance(est.steps[-1][1], Lasso)):
        print("❌ Aranan model Pipeline(StandardScaler, Lasso) olmalı: ölçekleyici her katta yalnızca o katın eğitim kısmıyla kurulmalı."); return
    if isinstance(arama.cv, KFold) or not isinstance(arama.cv, TimeSeriesSplit):
        print("❌ λ zamana saygılı çapraz doğrulamayla seçilmemiş. cv=TimeSeriesSplit(5) kullan; karıştırılmış KFold geleceği gösterir."); return
    if arama.scoring != "neg_mean_squared_error":
        print("❌ Seçim ölçüsünü açıkça yaz: scoring='neg_mean_squared_error' (HKO)."); return
    if not hasattr(arama, "best_estimator_"):
        print("❌ Arama tamamlanmamış ya da refit=False: seçilen λ ile tüm eğitim verisinde yeniden kurulmalı."); return
    son = arama.best_estimator_
    adlar = list(getattr(son, "feature_names_in_", []))
    if len(adlar) != 51:
        print("❌ Model 51 özellik adıyla (egitim[ozellikler]) kurulmalı."); return
    if egitim["hedef_ayi"].max() >= KESIM or test["hedef_ayi"].min() < KESIM:
        print("❌ 'egitim' ve 'test' hedef ayına göre bölünmemiş (Görev A)."); return
    if son[0].n_samples_seen_ != len(egitim) or not np.allclose(son[0].mean_, egitim[adlar].mean().values):
        print("❌ Son model yalnızca 'egitim' satırlarıyla kurulmamış: fit(egitim[ozellikler], egitim['hedef']) olmalı."); return
    if abs(_kh(arama.predict(test[adlar]), test["hedef"]) - float(kok_hko_lasso)) > 1e-6:
        print("❌ 'kok_hko_lasso', arama.predict(test[ozellikler]) ile test['hedef'] arasında hesaplanmalı (test satırlarındaki öngörüler)."); return
    r = _referans()
    if abs(float(kok_hko_lasso) - r["lasso"]) > 0.01:
        print("❌ Lasso'nun test kök HKO'su beklenenden farklı. λ ızgarası ({'lasso__alpha': np.logspace(-4, 0.5, 60)}) ve max_iter=50000 isteğe uygun mu?"); return
    nz = int(np.sum(np.abs(son[-1].coef_) > 0))
    if len(secilenler) != nz:
        print(f"❌ 'secilenler' {len(secilenler)} öğe içeriyor, ama sıfırdan farklı katsayı sayısı {nz}. Negatif katsayıları da say (coef_ != 0)."); return
    if "aylik_g11" not in secilenler:
        print("ℹ️ Bilgi: aylik_g11 (baz etkisi) listede yok; özellik adlarını kontrol et.")
    print(f"Seçilen λ: {arama.best_params_['lasso__alpha']:.4f} · seçilen değişken: {nz} / 51 · test satırı: {len(test)}")
    print(f"✅ Tüm kontroller geçti (Lasso ≈ {r['lasso']:.3f}, saf ≈ {r['saf']:.3f}). Tamamlama kodun: " + _k("TUwzLUxTTy03WFBE"))

print("G tablosu:", G.shape)
G.tail(3).round(2)

## Görev A — Problem ve kıyas ölçütü

1. **Özellikler:** `G`'nin `aylik` dışındaki her sütunu için 0, 1 ve 2 ay gecikmeli değerler (`{sütun}_g0`, `{sütun}_g1`, `{sütun}_g2`) ve `aylik`'in 0–11 ay gecikmeli değerleri (`aylik_g0` … `aylik_g11`): toplam 51.
2. **Hedef:** gelecek ayın yıllık enflasyonu, `hedef` adlı sütun: `G["enf"].shift(-1)`.
3. Özellikleri ve hedefi `tablo` adlı bir tabloda birleştir, `1984-01-01`'den başlat ve eksik değerli satırları at. Sonra hedefin ait olduğu ayı ekle: `tablo["hedef_ayi"] = tablo.index + pd.DateOffset(months=1)` (indeks özellik ayıdır).
4. **Hedef ayına göre böl:** `hedef_ayi` `pd.Timestamp("2015-01-01")`'den önce olan satırlar `egitim`, diğerleri `test`. Özellik adlarını `ozellikler` listesine yaz. Son eğitim hedefini ve ilk test hedefini yazdır.
5. Test döneminde **saf öngörünün** (`enf_g0`) ve eğitim verisiyle standartlaştırıp kurduğun **51 değişkenli EKK**'nın kök HKO'sunu hesapla: `kok_hko_saf`, `kok_hko_ekk`.

**Denetim soruları:** Hedef `shift(-1)` mi? Özelliklerde negatif kaydırma var mı? Kesim hedef ayına göre mi (Aralık 2014 satırının hedefi Ocak 2015'tir)? Tarih metninden `pd.Timedelta` çıkarılıyor mu (`TypeError`; önce `pd.Timestamp`)? Bölmede karıştırma var mı? Eksik aylar doldurulmuş mu?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(tablo, egitim, test, kok_hko_saf, kok_hko_ekk)

## Görev B — Lasso

1. `StandardScaler` ile `Lasso(max_iter=50000)`'yu `make_pipeline` ile bir `Pipeline`'a koy: ölçekleme her doğrulama katında yalnızca o katın eğitim kısmından öğrenilsin.
2. λ'yı `GridSearchCV(boru, {"lasso__alpha": np.logspace(-4, 0.5, 60)}, cv=TimeSeriesSplit(5), scoring="neg_mean_squared_error")` ile seç; arama nesnesini `arama` değişkenine ata ve `egitim[ozellikler]`, `egitim["hedef"]` ile `fit` et (arama bitince seçilen λ ile tüm eğitim verisinde yeniden kurulur).
3. `arama.predict(test[ozellikler])` ile test kök HKO'sunu hesapla, `kok_hko_lasso`'ya ata ve saf öngörüyle karşılaştır.
4. Son modelin (`arama.best_estimator_[-1]`) katsayısı sıfırdan farklı özelliklerini `secilenler` listesine yaz; mutlak değere göre ilk 8'ini katsayılarıyla yazdır.

**Denetim soruları:** Ölçekleyici katların içinde mi? `cv` ne? `scoring` açıkça HKO mu? Seçilen değişkenler iktisadi olarak ne anlatıyor (baz etkisi: `aylik_g11`)?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(arama, kok_hko_lasso, secilenler, egitim, test)

## Denetim notların

- Gemini hedefi ve bölmeyi nasıl kurdu? Değiştirmen gerekti mi?
- Ölçekleme ve çapraz doğrulama nasıl yapıldı?
- Lasso saf öngörüyü ne kadar geçiyor? Hangi veri kesitiyle (indirme zamanı, son hedef ayı)? Hangi değişkenler seçildi, ne anlama geliyor?
- Kontroller neyi sınamadı? (ör. yayın takvimi, veri revizyonları)